# Recipe 07 — Enrichers
> Augment context and queries with additional information during pipeline execution.

| | |
|---|---|
| **Module** | `anchor.pipeline.enrichment` |
| **Key classes** | `MemoryContextEnricher`, `ContextQueryEnricher` |
| **Difficulty** | Intermediate |

In [ ]:
from anchor.pipeline import ContextPipeline
from anchor.pipeline.enrichment import MemoryContextEnricher, ContextQueryEnricher
from anchor.models import QueryBundle, ContextItem, SourceType
from anchor.formatters import GenericTextFormatter
from anchor.memory import SlidingWindowMemory

## 1 — What are enrichers?
Enrichers inject additional signals into the pipeline:

- **MemoryContextEnricher** — injects conversation memory as context items.
- **ContextQueryEnricher** — rewrites or augments the query using existing context.

## 2 — MemoryContextEnricher
Automatically converts memory turns into `ContextItem` objects so they
compete for space alongside retrieved documents.

In [ ]:
memory = SlidingWindowMemory(max_tokens=1024)
memory.add_turn("user", "What is RAG?")
memory.add_turn("assistant", "RAG stands for Retrieval-Augmented Generation. "
                "It combines a retriever with a language model.")
memory.add_turn("user", "How does the retriever work?")

enricher = MemoryContextEnricher()
print(f"Enricher type: {type(enricher).__name__}")
print(f"Memory turns : {len(memory.get_turns())}")

## 3 — Use the enricher in a pipeline
Attach memory and the enricher. The enricher runs automatically during `build()`.

In [ ]:
pipeline = ContextPipeline(max_tokens=4096)
pipeline.add_system_prompt("You are a helpful assistant.")
pipeline.with_formatter(GenericTextFormatter())
pipeline.with_memory(memory)

query = QueryBundle(query_str="Can you explain the retriever component?")
result = pipeline.build(query)

print(f"Items in window : {len(result.window.items)}")
print(f"Token usage     : {result.window.used_tokens}")

## 4 — ContextQueryEnricher
Enriches the query by incorporating existing context — useful for
multi-hop retrieval where the refined query depends on what was already found.

In [ ]:
enricher = ContextQueryEnricher()
print(f"Enricher type: {type(enricher).__name__}")

## 5 — Combining enrichers
Enrichers can be layered: memory enrichment feeds context to the query
enricher for richer downstream retrieval.

In [ ]:
pipeline2 = ContextPipeline(max_tokens=4096)
pipeline2.add_system_prompt("You are a helpful assistant.")
pipeline2.with_formatter(GenericTextFormatter())
pipeline2.with_memory(memory)

query = QueryBundle(query_str="What about reranking?")
result = pipeline2.build(query)

print(f"Items     : {len(result.window.items)}")
print(f"Tokens    : {result.window.used_tokens}")
print(f"Utilization: {result.window.utilization:.1%}")

## 6 — Inspecting enriched items
Check which items came from memory versus retrieval by looking at `source`.

In [ ]:
for item in result.window.items:
    print(f"  {item.id:12s}  source={str(item.source):20s}  "
          f"tokens={item.token_count}")

## Key Takeaways
- `MemoryContextEnricher` converts memory turns into ranked context items.
- `ContextQueryEnricher` refines queries using existing context.
- Enrichers run automatically during `pipeline.build()` and compose freely.

**Next:** Return to the [cookbook index](../README.md)